## Cell 1 — Import libraries and configure model parameters

In [ ]:
import argparse, json, os, time, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
SEED = 42
N_SEEDS = 5
W_XGB = 0.70
TIME_ORDER = {"Early_Morning": 0, "Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4, "Late_Night": 5}
STOPS_ORDER = {"zero": 0, "one": 1, "two_or_more": 2}

XGB_PARAMS = dict(learning_rate=0.03624395751521409, max_depth=8,
    min_child_weight=6.681005468699639, subsample=0.7810250348662543,
    colsample_bytree=0.5043246873237734, gamma=4.456162588739199,
    reg_alpha=4.3468329984749845, reg_lambda=22.973497078303144,
    max_cat_to_onehot=4)
XGB_HUBER_SLOPE = 295.776275462723
XGB_ROUNDS = 1034
LGB_PARAMS = dict(learning_rate=0.03612471563647455, num_leaves=96,
    min_child_samples=14, subsample=0.8713370550415243,
    colsample_bytree=0.6696889490854012, reg_alpha=0.07430088704239915,
    reg_lambda=33.64523857814381, cat_smooth=4.319459793884871,
    cat_l2=22.698415206983707, min_split_gain=0.8831968480347183)
LGB_HUBER_DELTA = 296.977423485737
LGB_ROUNDS = 922

## Cell 2 — Build flight features
Creates numeric, categorical, route, timing, and flight-frequency features.

In [ ]:
def build_features(d, full):
    f = pd.DataFrame(index=d.index)
    f["duration"] = d["duration"].astype(float)
    f["days_left"] = d["days_left"].astype(float)
    f["stops_n"] = d["stops"].map(STOPS_ORDER).astype(float)
    f["dep_ord"] = d["departure_time"].map(TIME_ORDER).astype(float)
    f["arr_ord"] = d["arrival_time"].map(TIME_ORDER).astype(float)
    f["is_business"] = (d["flight_class"] == "Business").astype(int)
    f["flight_num"] = d["flight"].str.split().str[1].astype(float)
    f["overnight"] = (f["arr_ord"] < f["dep_ord"]).astype(int)
    f["dur_per_leg"] = f["duration"] / (f["stops_n"] + 1)
    f["last_week"] = (f["days_left"] <= 7).astype(int)
    f["last_3days"] = (f["days_left"] <= 3).astype(int)
    flight_counts = full["flight"].value_counts()
    f["flight_freq"] = d["flight"].map(flight_counts).astype(float)
    for c in ["airline", "source_city", "destination_city"]:
        f[c] = d[c].astype(str)
    f["route"] = d["source_city"].astype(str) + ">" + d["destination_city"].astype(str)
    f["route_class"] = f["route"] + "|" + d["flight_class"].astype(str)
    f["route_stops"] = f["route"] + "|" + d["stops"].astype(str)
    f["flight_class"] = d["flight_class"].astype(str)
    f["stops"] = d["stops"].astype(str)
    f["flight_key"] = d["flight"].astype(str) + "|" + d["flight_class"].astype(str)
    return f

## Cell 3 — Target encoding and ID-neighbor feature helpers
Adds out-of-fold flight-key target encoding and nearby-ID price/metadata features.

In [ ]:
def target_encode_flight(a, ya_log, others, smooth=10, n_splits=5, seed=SEED):
    prior = float(np.mean(ya_log))
    def stats(keys, values):
        return pd.DataFrame({"k": np.asarray(keys), "y": np.asarray(values)}).groupby("k").y.agg(["sum", "count"])
    def enc(keys, g):
        sums = pd.Series(np.asarray(keys)).map(g["sum"]).fillna(0).to_numpy()
        counts = pd.Series(np.asarray(keys)).map(g["count"]).fillna(0).to_numpy()
        return (sums + smooth * prior) / (counts + smooth)
    out = np.zeros(len(a), dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for i, j in kf.split(a):
        g = stats(a.flight_key.iloc[i].values, ya_log[i])
        out[j] = enc(a.flight_key.iloc[j].values, g)
    g_all = stats(a.flight_key.values, ya_log)
    transformed = a.copy()
    transformed["te_flight"] = out
    result_others = []
    for b in others:
        bb = b.copy()
        bb["te_flight"] = enc(bb.flight_key.values, g_all)
        result_others.append(bb)
    return transformed, result_others


def small(frame, ids):
    return pd.DataFrame({"id": np.asarray(ids),
        "airline": frame.airline.astype(str).to_numpy(),
        "stops_n": frame.stops_n.to_numpy(),
        "duration": frame.duration.to_numpy()})


def id_neighbour_features(T, log_prices, Q, self_in_T, K=2):
    order = np.argsort(T.id.to_numpy())
    s = T.iloc[order].reset_index(drop=True)
    ly = np.asarray(log_prices)[order]
    s_id = s.id.to_numpy()
    pos = np.searchsorted(s_id, Q.id.to_numpy())
    first_right = pos + (1 if self_in_T else 0)
    out = {}
    q_ids = Q.id.to_numpy()
    for k in range(1, K + 1):
        for side, idx in (("p", pos-k), ("n", first_right+k-1)):
            ok = (idx >= 0) & (idx < len(s))
            ci = np.clip(idx, 0, len(s)-1)
            out[f"id{side}{k}"] = np.where(ok, ly[ci], np.nan)
            out[f"id{side}g{k}"] = np.where(ok, np.abs(s_id[ci]-q_ids), np.nan)
            out[f"id{side}{k}_sa"] = np.where(ok, (s.airline.to_numpy()[ci] == Q.airline.to_numpy()).astype(float), np.nan)
            out[f"id{side}{k}_ss"] = np.where(ok, (s.stops_n.to_numpy()[ci] == Q.stops_n.to_numpy()).astype(float), np.nan)
            out[f"id{side}{k}_dd"] = np.where(ok, np.abs(s.duration.to_numpy()[ci]-Q.duration.to_numpy()), np.nan)
    d = pd.DataFrame(out)
    price_cols = [f"idp{k}" for k in range(1,K+1)] + [f"idn{k}" for k in range(1,K+1)]
    d["id_nb_mean"] = d[price_cols].mean(axis=1)
    d["id_nb_med"] = d[price_cols].median(axis=1)
    d["id_nb_mean1"] = d[["idp1", "idn1"]].mean(axis=1)
    d["id_nb_min"] = d[price_cols].min(axis=1)
    d["id_nb_max"] = d[price_cols].max(axis=1)
    return d.reset_index(drop=True)

## Cell 4 — Build model matrices and align categorical columns
Combines engineered features with neighbor features and makes category levels consistent.

In [ ]:
def make_matrices(F_tr, y_tr, ids_tr, F_queries, ids_queries):
    a, bs = target_encode_flight(F_tr.reset_index(drop=True), np.log1p(y_tr),
        [q.reset_index(drop=True) for q in F_queries])
    logy = np.log1p(y_tr)
    T = small(a, ids_tr)
    a = pd.concat([a.reset_index(drop=True), id_neighbour_features(T, logy, T, True)], axis=1)
    out_bs = []
    for b, ids in zip(bs, ids_queries):
        q = small(b, ids)
        nb = id_neighbour_features(T, logy, q, False)
        out_bs.append(pd.concat([b.reset_index(drop=True), nb], axis=1))
    # All models use numeric/categorical columns except the raw key used for encoding.
    return a, out_bs


def prep_categories(A, Bs):
    cat_cols = ["airline", "source_city", "destination_city", "route", "route_class", "route_stops", "flight_class", "stops"]
    for c in cat_cols:
        # Use a shared category universe between train and query matrices.
        vals = pd.concat([A[c].astype(str)] + [B[c].astype(str) for B in Bs], ignore_index=True)
        cats = pd.Index(vals.unique())
        A[c] = pd.Categorical(A[c].astype(str), categories=cats)
        for B in Bs:
            B[c] = pd.Categorical(B[c].astype(str), categories=cats)
    return A, Bs


def model_columns(A):
    return [c for c in A.columns if c != "flight_key"]

## Cell 5 — Model training and evaluation helper functions
Defines XGBoost/LightGBM prediction functions and evaluation metrics.

In [ ]:
def fit_predict_xgb(A, y, B, cols, seeds, rounds, n_jobs=4):
    preds=[]
    for s in seeds:
        m = xgb.XGBRegressor(**XGB_PARAMS, objective="reg:pseudohubererror", huber_slope=XGB_HUBER_SLOPE,
            n_estimators=int(rounds), tree_method="hist", enable_categorical=True,
            random_state=int(s), n_jobs=n_jobs, verbosity=0)
        m.fit(A[cols], y)
        preds.append(m.predict(B[cols]))
    return np.clip(np.mean(preds, axis=0), 0, None)


def fit_predict_lgb(A, y, B, cols, seeds, rounds, n_jobs=4):
    preds=[]
    for s in seeds:
        m = lgb.LGBMRegressor(**LGB_PARAMS, objective="huber", alpha=LGB_HUBER_DELTA,
            n_estimators=int(rounds), subsample_freq=1, random_state=int(s), verbose=-1, n_jobs=n_jobs)
        m.fit(A[cols], y, categorical_feature="auto")
        preds.append(m.predict(B[cols]))
    return np.clip(np.mean(preds, axis=0), 0, None)


def metric_row(name, y_true, pred):
    return {"model": name, "R2_all": r2_score(y_true, pred),
        "R2_price_lt_4000": r2_score(y_true[y_true < 4000], pred[y_true < 4000]),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MAE": mean_absolute_error(y_true, pred)}

## Cell 6 — Locate data files and load train/test data
Change `DATA_DIR` manually if your CSV files are stored elsewhere.

In [ ]:
import glob
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

candidates = [os.getcwd(), "/mnt/data"] + glob.glob("/kaggle/input/*")
DATA_DIR = next(
    (d for d in candidates
     if os.path.exists(os.path.join(d, "train.csv"))
     and os.path.exists(os.path.join(d, "test.csv"))),
    None
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv. Set DATA_DIR to their folder."
    )

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

needed = {
    "id", "airline", "flight", "source_city", "departure_time", "stops",
    "arrival_time", "destination_city", "flight_class", "duration",
    "days_left", "price"
}
if not needed.issubset(train.columns):
    raise ValueError(f"train.csv missing columns: {needed - set(train.columns)}")
if "id" not in test.columns:
    raise ValueError("test.csv must contain an 'id' column.")
if not set(train.id).isdisjoint(set(test.id)):
    raise ValueError("Train and test IDs overlap.")

y = train["price"].to_numpy(dtype=float)
ids = train["id"].to_numpy()

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")

## Cell 7 — Engineer features for train and test
Builds features using the combined train/test rows, then splits them back into train and test feature tables.

In [ ]:
full = pd.concat([train.drop(columns=["price"]), test], ignore_index=True)
F_all = build_features(full, full)

cat_cols = [
    "airline", "source_city", "destination_city", "route",
    "route_class", "route_stops", "flight_class", "stops"
]
for c in cat_cols:
    F_all[c] = F_all[c].astype("category")

F_train = F_all.iloc[:len(train)].reset_index(drop=True)
F_test = F_all.iloc[len(train):].reset_index(drop=True)

both = pd.concat([
    train.assign(_split="tr"),
    test.assign(_split="te")
]).sort_values("id")
adjacent_same = (
    both.flight_class.astype(str).to_numpy()[1:]
    == both.flight_class.astype(str).to_numpy()[:-1]
).mean()

train_sorted = train.sort_values("id")
log_prices_sorted = np.log1p(train_sorted.price.to_numpy())
adjacent_corr = np.corrcoef(log_prices_sorted[:-1], log_prices_sorted[1:])[0, 1]

print(
    "ID structure check: "
    f"combined IDs sequential={np.array_equal(both.id.to_numpy(), np.arange(len(both)))}; "
    f"adjacent class match={adjacent_same:.3f}; "
    f"train log-price adjacent correlation={adjacent_corr:.3f}"
)

## Cell 8 — Validate models on a holdout set (optional but recommended)
Compares an XGBoost baseline without ID-neighbor features against XGBoost, LightGBM, and their blend with neighbor features. Writes holdout metrics and predictions.

In [ ]:
RUN_VALIDATION = True  # Set to False to skip holdout evaluation.

if RUN_VALIDATION:
    bins = pd.qcut(y, 10, labels=False, duplicates="drop")
    i_dev, i_hold = train_test_split(
        np.arange(len(y)),
        test_size=0.2,
        random_state=SEED,
        stratify=bins
    )

    A_dev, (B_hold,) = make_matrices(
        F_train.iloc[i_dev], y[i_dev], ids[i_dev],
        [F_train.iloc[i_hold]], [ids[i_hold]]
    )
    A_dev, [B_hold] = prep_categories(A_dev, [B_hold])

    cols_all = model_columns(A_dev)
    cols_noid = [c for c in cols_all if not c.startswith("id")]

    print(
        f"Holdout fit: {len(i_dev)} train rows, {len(i_hold)} validation rows, "
        f"{len(cols_all)} total model features; "
        f"{len(cols_all) - len(cols_noid)} neighbor features"
    )

    t0 = time.time()
    p_noid = fit_predict_xgb(
        A_dev, y[i_dev], B_hold, cols_noid,
        [SEED], int(XGB_ROUNDS * 1.1), n_jobs=4
    )
    print(f"XGBoost baseline without ID-neighbor features: {time.time() - t0:.1f}s")

    p_x = fit_predict_xgb(
        A_dev, y[i_dev], B_hold, cols_all,
        [SEED, SEED + 1, SEED + 2], int(XGB_ROUNDS * 1.1), n_jobs=4
    )
    print(f"XGBoost with ID-neighbor features finished: {time.time() - t0:.1f}s")

    p_l = fit_predict_lgb(
        A_dev, y[i_dev], B_hold, cols_all,
        [SEED, SEED + 1, SEED + 2], int(LGB_ROUNDS * 1.1), n_jobs=4
    )
    print(f"LightGBM with ID-neighbor features finished: {time.time() - t0:.1f}s")

    p_blend = W_XGB * p_x + (1 - W_XGB) * p_l
    metrics = pd.DataFrame([
        metric_row("XGBoost without ID-neighbor features", y[i_hold], p_noid),
        metric_row("XGBoost with ID-neighbor features", y[i_hold], p_x),
        metric_row("LightGBM with ID-neighbor features", y[i_hold], p_l),
        metric_row("70% XGBoost + 30% LightGBM with ID-neighbor features", y[i_hold], p_blend)
    ])

    metrics_path = os.path.join(OUT_DIR, "flight_price_id_neighbor_holdout_metrics.csv")
    metrics.to_csv(metrics_path, index=False)
    display(metrics.round(5))

    holdout_predictions = pd.DataFrame({
        "id": ids[i_hold],
        "actual_price": y[i_hold],
        "pred_noid_xgb": p_noid,
        "pred_id_xgb": p_x,
        "pred_id_lgb": p_l,
        "pred_id_blend": p_blend
    })
    holdout_path = os.path.join(OUT_DIR, "flight_price_id_neighbor_holdout_predictions.csv")
    holdout_predictions.to_csv(holdout_path, index=False)

    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved holdout predictions to: {holdout_path}")
else:
    print("Holdout validation skipped.")

## Cell 9 — Train final models on all labeled data
Rebuilds target-encoding and neighbor features using all labeled rows, then trains both models over multiple random seeds.

In [ ]:
N_FINAL_SEEDS = 5
N_JOBS = 4

print("Building full-data neighbor / target-encoding matrices...")
A_full, [B_test] = make_matrices(
    F_train, y, ids,
    [F_test], [test.id.to_numpy()]
)
A_full, [B_test] = prep_categories(A_full, [B_test])
cols = model_columns(A_full)
seeds = [SEED + i for i in range(max(1, N_FINAL_SEEDS))]

t0 = time.time()
pred_x = fit_predict_xgb(
    A_full, y, B_test, cols, seeds,
    int(XGB_ROUNDS * 1.25), n_jobs=N_JOBS
)
print(f"Full-data XGBoost finished ({len(seeds)} seeds) in {time.time() - t0:.1f}s")

pred_l = fit_predict_lgb(
    A_full, y, B_test, cols, seeds,
    int(LGB_ROUNDS * 1.25), n_jobs=N_JOBS
)
print(f"Full-data LightGBM finished ({len(seeds)} seeds) in {time.time() - t0:.1f}s")

pred = W_XGB * pred_x + (1 - W_XGB) * pred_l
print("Final blend created using weights:", W_XGB, "for XGBoost and", 1 - W_XGB, "for LightGBM")

## Cell 10 — Save submission, model predictions, and run summary
Creates the submission CSV plus diagnostic predictions and a JSON summary.

In [ ]:
submission = pd.DataFrame({
    "id": test.id.to_numpy(),
    "price": np.round(np.clip(pred, 0, None), 2)
})
submission_path = os.path.join(OUT_DIR, "submission_id_neighbor_xgb_lgbm.csv")
submission.to_csv(submission_path, index=False)

test_predictions = pd.DataFrame({
    "id": test.id.to_numpy(),
    "xgb_id_neighbor_price": pred_x,
    "lgb_id_neighbor_price": pred_l,
    "blend_price": pred
})
test_predictions_path = os.path.join(
    OUT_DIR, "flight_price_id_neighbor_test_predictions.csv"
)
test_predictions.to_csv(test_predictions_path, index=False)

summary = {
    "method": "XGBoost + LightGBM + ID-neighbor price features",
    "xgb_weight": W_XGB,
    "lgb_weight": 1 - W_XGB,
    "seeds": seeds,
    "xgb_rounds": int(XGB_ROUNDS * 1.25),
    "lgb_rounds": int(LGB_ROUNDS * 1.25),
    "n_train": len(train),
    "n_test": len(test),
    "output": "submission_id_neighbor_xgb_lgbm.csv",
    "caveat": (
        "Uses labels of neighboring training IDs as features. Valid only for "
        "this interleaved train/test row-order setup."
    )
}
summary_path = os.path.join(OUT_DIR, "flight_price_id_neighbor_run_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

assert len(submission) == len(test)
assert submission["id"].is_unique
assert np.array_equal(submission["id"].to_numpy(), test["id"].to_numpy())
assert submission["price"].notna().all()
assert (submission["price"] >= 0).all()

print(f"Saved submission: {submission_path}")
print(f"Submission shape: {submission.shape}")
print(f"Predicted price range: {submission.price.min():.2f} – {submission.price.max():.2f}")
print(f"Saved test predictions: {test_predictions_path}")
print(f"Saved run summary: {summary_path}")